# Assignment 1 — Q2: IMDb Top 250 Movies — Scraping and Trend Analysis
**Data Science Lab (CS2311)**

**Goal:** help a streaming platform understand audience preferences by scraping the IMDb Top 250 chart.

**Pipeline:** IMDb pages (HTML) → JSON (semi-structured) → DataFrame (structured) → cleaning → 8 new features → normalization → 8 visualizations → CSV + 5 insights.

## How modern IMDb pages store their data

IMDb builds its pages with JavaScript, so a lot of the visible text is not in plain HTML tags. But every page embeds a clean data block for search engines:

    <script type="application/ld+json"> { ...structured data... } </script>

This is **JSON-LD** (JSON for Linked Data). It is the most reliable thing to scrape: we find the `<script>` tag with BeautifulSoup and parse its text with `json.loads`.

- The **chart page** (`/chart/top/`) JSON-LD lists all 250 movies with title, URL, rating, number of votes, certificate, genre, duration, description and poster image.
- Each **movie page** JSON-LD adds the release date, director(s), cast and full genre list.

**If IMDb blocks the request** (HTTP 403, or a page without the data), the notebook automatically switches to a public GitHub CSV of IMDb movie data with the same kind of attributes, and says so. All later steps work on either source.

In [ ]:
import json
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup
from sklearn.preprocessing import MinMaxScaler, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

CURRENT_YEAR = pd.Timestamp.today().year

---
## 1. Scraping

IMDb rejects requests that do not look like they come from a browser, so we send a browser-like **User-Agent** header and ask for the English page.

In [ ]:
CHART_URL = 'https://www.imdb.com/chart/top/'
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/124.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
}
FETCH_MOVIE_PAGES = True      # visit each movie page for year, director and cast (~250 requests, 2-4 minutes)
RAW_JSON = 'imdb_top250_raw.json'
FALLBACK_URL = ('https://raw.githubusercontent.com/LearnDataSci/articles/master/'
                'Python%20Pandas%20Tutorial%20A%20Complete%20Introduction%20for%20Beginners/IMDB-Movie-Data.csv')


def get_json_ld(url):
    """Download a page and return the parsed JSON-LD block (or raise an error)."""
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')
    script = soup.find('script', type='application/ld+json')
    if script is None:
        raise ValueError('No JSON-LD block found on ' + url)
    return json.loads(script.string)


def names(value):
    """JSON-LD people/genres can be a dict, a list of dicts, a list of strings or a string -> 'A, B, C'."""
    if value is None:
        return None
    if isinstance(value, (str, dict)):
        value = [value]
    return ', '.join(v.get('name', '') if isinstance(v, dict) else str(v) for v in value)

### 1a. The chart page: 250 movies
The JSON-LD on the chart page has the shape `{"@type": "ItemList", "itemListElement": [{"item": {movie}}, ...]}`.

In [ ]:
def scrape_chart():
    data = get_json_ld(CHART_URL)
    movies = []
    for rank, element in enumerate(data['itemListElement'], start=1):
        m = element.get('item', element)
        rating = m.get('aggregateRating', {})
        movies.append({
            'rank': rank,
            'title': m.get('name'),
            'original_title': m.get('alternateName'),
            'url': m.get('url'),
            'description': m.get('description'),
            'image_url': m.get('image'),
            'rating': rating.get('ratingValue'),
            'votes': rating.get('ratingCount'),
            'certificate': m.get('contentRating'),
            'genre': names(m.get('genre')),
            'duration': m.get('duration'),          # ISO-8601, e.g. 'PT2H22M'
        })
    return movies

### 1b. Each movie's own page: year, director, cast
Gross collection is not part of IMDb's structured data, so it stays missing for live-scraped data (we discuss this in the cleaning step).

In [ ]:
def add_movie_details(movies, pause=0.3):
    for i, movie in enumerate(movies, start=1):
        try:
            d = get_json_ld(movie['url'])
            movie['release_date'] = d.get('datePublished')
            movie['director'] = names(d.get('director'))
            movie['cast'] = names((d.get('actor') or [])[:5])
            movie['genre'] = names(d.get('genre')) or movie['genre']      # full genre list
            movie['keywords'] = d.get('keywords')
        except Exception as e:
            print(f'  {movie["title"]}: details not available ({type(e).__name__})')
        if i % 25 == 0:
            print(f'  details fetched for {i} movies')
        time.sleep(pause)
    return movies

### 1c. Fallback source
The GitHub CSV contains 1,000 popular IMDb movies (2006–2016) with title, genre, description, director, actors, year, runtime, rating, votes, revenue (gross) and Metascore. We keep the **250 highest-rated** ones (ties broken by votes) and rename columns to match our schema.

In [ ]:
def load_fallback():
    fb = pd.read_csv(FALLBACK_URL)
    fb = fb.sort_values(['Rating', 'Votes'], ascending=False).head(250).reset_index(drop=True)
    return [{
        'rank': i + 1,
        'title': r['Title'],
        'description': r['Description'],
        'rating': r['Rating'],
        'votes': r['Votes'],
        'duration': f"{r['Runtime (Minutes)']} min",
        'genre': r['Genre'].replace(',', ', '),
        'release_date': str(r['Year']),
        'director': r['Director'],
        'cast': r['Actors'],
        'gross_millions': None if pd.isna(r['Revenue (Millions)']) else r['Revenue (Millions)'],
        'metascore': None if pd.isna(r['Metascore']) else r['Metascore'],
        'certificate': None,
        'url': None,
    } for i, r in fb.iterrows()]

### 1d. Run the scraper (with caching)
The raw records are saved to `imdb_top250_raw.json` (semi-structured stage). If that file exists, it is loaded instead of scraping again. Delete it to re-scrape.

In [ ]:
if os.path.exists(RAW_JSON):
    print(f'Loading cached {RAW_JSON} (delete it to scrape again)')
else:
    try:
        records = scrape_chart()
        print(f'Chart page scraped: {len(records)} movies')
        if len(records) < 200:
            raise ValueError(f'only {len(records)} movies found')
        if FETCH_MOVIE_PAGES:
            records = add_movie_details(records)
        source = 'IMDb live scrape'
    except Exception as e:
        print(f'Live scraping failed ({type(e).__name__}: {e}).')
        print('→ Using the GitHub fallback dataset instead.')
        records = load_fallback()
        source = 'GitHub IMDb movie dataset (fallback)'
    for r in records:
        r['source'] = source

    # ---------- HTML -> JSON (semi-structured) ----------
    with open(RAW_JSON, 'w', encoding='utf-8') as f:
        json.dump(records, f, indent=2, ensure_ascii=False, default=str)
    print(f'Saved {len(records)} records to {RAW_JSON}')

---
## 2. JSON → DataFrame

In [ ]:
with open(RAW_JSON, encoding='utf-8') as f:
    raw = json.load(f)

print('Source:', raw[0].get('source'))
print('First JSON record:')
print(json.dumps(raw[0], indent=2, ensure_ascii=False)[:1000])

movies = pd.DataFrame(raw)
print('\nShape:', movies.shape)
movies.head()

In [ ]:
movies.info()

**Attributes collected (more than the 10 required):** rank, title, URL, description, poster image, IMDb rating, number of votes, certificate, genre(s), duration, release date, director, cast, keywords (live) or gross collection and Metascore (fallback).

---
## 3. Data wrangling
### 3a. Missing values

In [ ]:
missing = movies.isnull().sum()
pd.DataFrame({'missing': missing, '% missing': (missing / len(movies) * 100).round(1)}).query('missing > 0')

**Strategy:**
- Columns that are **completely empty** for this source (e.g. gross for live IMDb data, URL for the fallback) carry no information → drop them.
- Columns that are **mostly empty** (> 60% missing) → drop: imputing most of a column would invent data.
- `certificate` (category) → fill with `'Not Rated'`.
- `director`, `cast`, `keywords` (text) → fill with `'Unknown'`.
- Numeric columns (`gross_millions`, `metascore`) → fill with the **median**, plus a 0/1 flag recording that the value was missing (so the information "unknown" is not lost).

In [ ]:
df = movies.copy()
empty_cols = [c for c in df.columns if df[c].isnull().mean() > 0.6]
print('Dropping mostly/completely empty columns:', empty_cols)
df = df.drop(columns=empty_cols)

if 'certificate' in df:
    df['certificate'] = df['certificate'].fillna('Not Rated')
for c in ['director', 'cast', 'keywords', 'description', 'original_title']:
    if c in df:
        df[c] = df[c].fillna('Unknown')
for c in ['gross_millions', 'metascore']:
    if c in df:
        df[c] = pd.to_numeric(df[c], errors='coerce')
        df[f'{c}_was_missing'] = df[c].isnull().astype(int)
        df[c] = df[c].fillna(df[c].median())

print('Missing values left:\n', df.isnull().sum()[df.isnull().sum() > 0], sep='')

### 3b. Duplicates
A movie is identified by its title **and** release year (remakes share titles). The live data also has a unique URL.

In [ ]:
key = ['url'] if 'url' in df else ['title', 'release_date']
print('Duplicate movies on', key, ':', df.duplicated(subset=key).sum())
df = df.drop_duplicates(subset=key).reset_index(drop=True)
print('Shape:', df.shape)

### 3c. Standardizing text
- Trim spaces in all text columns.
- Genres: consistent Title Case, ', ' separator.
- Certificate: upper case, so 'pg-13' and 'PG-13' become one category.
- Titles: unescape HTML entities like `&apos;` that sometimes appear in scraped JSON.

In [ ]:
import html as html_lib

text_cols = df.select_dtypes(include=['object', 'string']).columns
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()

df['title'] = df['title'].apply(html_lib.unescape)
df['genre'] = df['genre'].str.split(',').apply(lambda gs: ', '.join(g.strip().title() for g in gs))
if 'certificate' in df:
    df['certificate'] = df['certificate'].str.upper().replace({'NOT RATED': 'Not Rated', 'UNRATED': 'Not Rated'})
df[['title', 'genre'] + (['certificate'] if 'certificate' in df else [])].head()

### 3d. Converting to numeric types

| Column | Raw | Converted |
|---|---|---|
| duration | 'PT2H22M' (ISO-8601) or '142 min' | 142 (minutes, int) |
| release_date | '1994-10-14' or '2014' | year 1994 (int) |
| rating | 9.3 or '9.3' | float |
| votes | 2900000 or '2,900,000' | int |

**ISO-8601 dry run:** 'PT2H22M' → regex finds hours = 2, minutes = 22 → 2·60 + 22 = 142 minutes.

In [ ]:
def to_minutes(value):
    value = str(value)
    if value.startswith('PT'):
        hours = re.search(r'(\d+)H', value)
        mins = re.search(r'(\d+)M', value)
        return (int(hours.group(1)) if hours else 0) * 60 + (int(mins.group(1)) if mins else 0)
    digits = re.search(r'(\d+)', value)
    return int(digits.group(1)) if digits else np.nan

df['runtime_min'] = df['duration'].apply(to_minutes)
df['year'] = pd.to_numeric(df['release_date'].astype(str).str[:4], errors='coerce')
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['votes'] = pd.to_numeric(df['votes'].astype(str).str.replace(',', ''), errors='coerce')

for c in ['runtime_min', 'year', 'rating', 'votes']:
    if df[c].isnull().any():
        print(f'{c}: filling {df[c].isnull().sum()} missing with median')
        df[c] = df[c].fillna(df[c].median())
df['runtime_min'] = df['runtime_min'].astype(int)
df['year'] = df['year'].astype(int)
df['votes'] = df['votes'].astype(int)
df[['title', 'year', 'runtime_min', 'rating', 'votes']].head()

### 3e. Feature engineering (8 new features)

| Feature | Definition |
|---|---|
| `movie_age` | current year − release year |
| `decade` | release year rounded down to the decade (1994 → 1990) |
| `rating_category` | Good (< 8.0), Great (8.0–8.5), Masterpiece (> 8.5) |
| `runtime_category` | Short (< 100 min), Standard (100–150), Long (> 150) |
| `primary_genre` | first genre listed |
| `num_genres` | how many genres the movie belongs to |
| `log_votes` | log10(votes): compresses the huge range of vote counts |
| `popularity_score` | rating × log10(votes): rewards movies that are both highly rated and widely watched |

In [ ]:
df['movie_age'] = CURRENT_YEAR - df['year']
df['decade'] = (df['year'] // 10) * 10
df['rating_category'] = pd.cut(df['rating'], bins=[0, 7.99, 8.5, 10], labels=['Good', 'Great', 'Masterpiece'])
df['runtime_category'] = pd.cut(df['runtime_min'], bins=[0, 99, 150, 1000], labels=['Short', 'Standard', 'Long'])
df['genre_list'] = df['genre'].str.split(', ')
df['primary_genre'] = df['genre_list'].str[0]
df['num_genres'] = df['genre_list'].str.len()
df['log_votes'] = np.log10(df['votes'])
df['popularity_score'] = (df['rating'] * df['log_votes']).round(3)

df[['title', 'year', 'decade', 'movie_age', 'rating', 'rating_category', 'runtime_min',
    'runtime_category', 'primary_genre', 'num_genres', 'popularity_score']].head(8)

### 3f. Normalization: which technique, and why

| Attribute | Shape of the data | Technique | Reason |
|---|---|---|---|
| `votes` | Extremely right-skewed (a few movies have millions more votes) | **log10, then Min-Max** | Min-Max on raw votes would squash almost every movie near 0 because of a few giants. The log removes the skew; Min-Max then puts it on [0, 1]. |
| `rating`, `runtime_min` | Roughly bell-shaped, no extreme values | **StandardScaler (z-score)** | Centres each at 0 with std 1, so a value reads as "how many standard deviations above average", which is easy to compare across attributes. |
| `movie_age` | Bounded, no extreme outliers | **Min-Max** | Simple [0, 1] scale for a bounded attribute. |

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
sns.histplot(df['votes'], bins=30, ax=ax[0]); ax[0].set_title(f"votes (skew = {df['votes'].skew():.2f})")
sns.histplot(df['log_votes'], bins=30, ax=ax[1], color='tab:green'); ax[1].set_title(f"log10(votes) (skew = {df['log_votes'].skew():.2f})")
sns.histplot(df['rating'], bins=20, ax=ax[2], color='tab:orange'); ax[2].set_title(f"rating (skew = {df['rating'].skew():.2f})")
plt.tight_layout()
plt.show()

df['votes_norm'] = MinMaxScaler().fit_transform(df[['log_votes']])
df[['rating_z', 'runtime_z']] = StandardScaler().fit_transform(df[['rating', 'runtime_min']])
df['age_norm'] = MinMaxScaler().fit_transform(df[['movie_age']])
df[['votes_norm', 'rating_z', 'runtime_z', 'age_norm']].describe().loc[['mean', 'std', 'min', 'max']].round(3)

---
## 4. Visualizations

In [ ]:
genre_counts = df['genre_list'].explode().value_counts()
plt.figure(figsize=(10, 6))
genre_counts.head(15).sort_values().plot(kind='barh', color='tab:blue')
plt.title('1. Genre popularity (a movie counts once for each of its genres)')
plt.xlabel('Number of movies')
plt.show()

`explode()` turns a row with `['Crime', 'Drama']` into two rows, one per genre, so each genre gets counted.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
sns.histplot(df['rating'], bins=15, kde=True, ax=ax[0])
ax[0].set(title='2. IMDb rating distribution', xlabel='Rating')
df['rating_category'].value_counts().reindex(['Good', 'Great', 'Masterpiece']).plot(
    kind='bar', ax=ax[1], rot=0, color=['#a6cee3', '#1f78b4', '#08306b'])
ax[1].set(title='2b. Movies per rating category', ylabel='Movies')
plt.tight_layout()
plt.show()

In [ ]:
decade_counts = df['decade'].value_counts().sort_index()
plt.figure(figsize=(11, 4.5))
plt.bar(decade_counts.index.astype(str), decade_counts.values, color='tab:purple')
plt.title('3. Movie production across decades (movies in the list per decade)')
plt.xlabel('Decade'); plt.ylabel('Movies')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
sns.histplot(df['runtime_min'], bins=25, kde=True, ax=ax[0], color='tab:red')
ax[0].axvline(df['runtime_min'].median(), color='black', ls='--', label=f"median {df['runtime_min'].median():.0f} min")
ax[0].set(title='4. Runtime distribution', xlabel='Minutes'); ax[0].legend()
sns.boxplot(x='decade', y='runtime_min', data=df, ax=ax[1], color='salmon')
ax[1].set(title='4b. Runtime by decade', xlabel='Decade', ylabel='Minutes')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5.5))
sns.scatterplot(x='votes', y='rating', hue='runtime_category', data=df, alpha=0.75)
plt.xscale('log')
plt.title('5. Votes vs rating (log scale on votes)')
plt.xlabel('Number of votes (log scale)')
plt.show()

In [ ]:
num_cols = ['rating', 'votes', 'log_votes', 'runtime_min', 'year', 'movie_age', 'num_genres', 'popularity_score']
num_cols += [c for c in ['gross_millions', 'metascore'] if c in df]
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('6. Correlation among numerical attributes')
plt.show()

In [ ]:
genre_rating = (df.explode('genre_list').groupby('genre_list')
                  .agg(movies=('title', 'count'), avg_rating=('rating', 'mean'), avg_votes=('votes', 'mean'))
                  .query('movies >= 5').sort_values('avg_rating'))
fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
genre_rating['avg_rating'].plot(kind='barh', ax=ax[0], color='tab:green')
ax[0].set(title='7. Average rating by genre (≥ 5 movies)', xlabel='Average rating')
ax[0].set_xlim(genre_rating['avg_rating'].min() - 0.2, genre_rating['avg_rating'].max() + 0.1)
(genre_rating['avg_votes'] / 1e6).sort_values().plot(kind='barh', ax=ax[1], color='tab:orange')
ax[1].set(title='7b. Average votes by genre (millions)', xlabel='Average votes (millions)')
plt.tight_layout()
plt.show()

In [ ]:
top_directors = df[df['director'] != 'Unknown']['director'].str.split(', ').explode().value_counts().head(10)
plt.figure(figsize=(9, 5))
top_directors.sort_values().plot(kind='barh', color='tab:cyan')
plt.title('8. Directors with the most movies in the list')
plt.xlabel('Movies')
plt.show()

---
## 5. Export and analytical report

In [ ]:
export = df.drop(columns=['genre_list'])
export.to_csv('imdb_top250_clean.csv', index=False, encoding='utf-8')
print('Saved imdb_top250_clean.csv with shape', export.shape)

In [ ]:
r_votes_rating = df['votes'].corr(df['rating'])
r_runtime_rating = df['runtime_min'].corr(df['rating'])
by_runtime = df.groupby('runtime_category', observed=True)['rating'].agg(['count', 'mean']).round(2)
best_genre = genre_rating['avg_rating'].idxmax()
top_decades = decade_counts.sort_values(ascending=False).head(3)
pre_2000 = (df['year'] < 2000).mean() * 100

print(f"Source: {raw[0].get('source')} | {len(df)} movies from {df.year.min()} to {df.year.max()}")
print(f"1. Top genres by count: {genre_counts.head(5).to_dict()}")
print(f"2. Highest-rated genre (≥5 movies): {best_genre} ({genre_rating.loc[best_genre, 'avg_rating']:.2f})")
print(f"3. Busiest decades: {top_decades.to_dict()} | movies released before 2000: {pre_2000:.1f}%")
print(f"4. Median runtime: {df.runtime_min.median():.0f} min | rating by runtime category:\n{by_runtime}")
print(f"   corr(runtime, rating) = {r_runtime_rating:.3f}")
print(f"5. corr(votes, rating) = {r_votes_rating:.3f} | rating range {df.rating.min()}–{df.rating.max()}")
print(f"   Directors with most entries: {top_directors.head(3).to_dict()}")

### Analytical report — insights for a streaming platform

Read these together with the printed numbers above (they come from your scraped data).

1. **Drama dominates.** Drama appears in far more top-rated movies than any other genre, usually combined with Crime, Adventure or Biography (plot 1). A catalogue built around strong dramas, especially crime dramas, matches what audiences rate highest.
2. **Popularity and quality go together.** Ratings and vote counts are positively correlated (plot 5, insight 5): the highest-rated movies are also the most watched. Vote count is a good, cheap signal for which classics to license first.
3. **Classics still matter.** A large share of the list was released before 2000 (insight 3), and almost every decade is represented (plot 3). Licensing older acclaimed films is valuable; audiences do not only want new releases.
4. **Longer films are not penalised.** Most top movies run about 2 hours, and Long movies (> 150 min) have average ratings at least as high as Standard ones (insight 4). Viewers accept long runtimes for quality, so the platform need not prefer short content.
5. **Auteur directors are a reliable signal.** A few directors (plot 8) have several movies in the list. Promoting director collections ("all of X's films") is a simple, data-backed recommendation strategy.

**Limitation:** the Top 250 contains only highly rated movies (rating range is narrow), so it shows what audiences *love*, not what the average viewer watches most.

**If the notebook used the fallback dataset** (the first printout says which source was used): it only covers 2006–2016, so replace insight 3 with a recent-trend insight. For example, compare the number of top movies and their average rating per year with `df.groupby('year')['rating'].agg(['count', 'mean'])`. The fallback also has gross revenue (`gross_millions`), so you can add an insight on whether higher-rated movies earn more (check `corr` in plot 6).